# Classify Intents with an LLM, and Compare

Yesterday, classic models, *embeddings* and a fine-tuned encoder classified MASSIVE's requests. Today, an LLM, without training:

1. Zero-shot: the list of described intents, a constrained output
2. With a few examples in the prompt
3. Compare every method
4. Read the errors

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data and Model

Yesterday's results: upload your files `predictions-en.csv` and `timings-en.csv` (*Files* panel on the left), or use ours, which the next cell downloads if they're missing. Ollama installs and downloads the model in the background.

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s massive-intents
!test -f predictions-en.csv || cp massive-intents/lab-results/predictions-en.csv .
!test -f timings-en.csv || cp massive-intents/lab-results/timings-en.csv .
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import create_model
from sklearn.metrics import f1_score

train = pd.read_parquet("massive-intents/en-train.parquet")
intents = pd.read_csv("massive-intents/intents.csv")
test = pd.read_parquet("massive-intents/en-test.parquet")
# The requests' texts, by their id.
predictions = pd.read_csv("predictions-en.csv", dtype={"id": str}).merge(test[["id", "text"]], on="id")
timings = pd.read_csv("timings-en.csv")
# 300 test requests, drawn at random: an LLM is too slow for all 2,974.
sample = predictions.sample(300, random_state=0).reset_index(drop=True)
labels = intents.label.tolist()
print(len(labels), "intents")

pull.wait()  # the model is downloaded
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# A first call loads the model onto the GPU, where it stays: before any parallel work.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1)
print(MODEL, "ready")

## 1. Zero-Shot

The output schema, `Intent`, has a single field, `intent`, whose value is necessarily one of the 60 labels: the model can't answer outside the list.

Write the system prompt: the task, then the list of intents with their description (`intents.description_en`). Write `classify(text, system)` that returns the chosen intent (`client.chat.completions.parse(..., response_format=Intent, temperature=0)`), classify `sample`'s 300 requests in parallel (`ThreadPoolExecutor(8)`), and measure the macro F1 and the time per text.

In [ ]:
Intent = create_model("Intent", intent=(Literal[tuple(labels)], ...))

In [ ]:
# Your code here

### Solution

In [ ]:
LIST = "\n".join(f"- {row.label}: {row.description_en}" for row in intents.itertuples())
ZERO_SHOT = f"""You classify requests made to a voice assistant by their intent.
Choose exactly one intent from this list:
{LIST}"""


def classify(text: str, system: str) -> str:
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": text}],
        response_format=Intent,
        temperature=0,
    )
    return response.choices[0].message.parsed.intent


def run(system: str) -> tuple[list[str], float]:
    start = time.time()
    with ThreadPoolExecutor(8) as pool:
        found = list(pool.map(lambda text: classify(text, system), sample.text))
    return found, time.time() - start


zero_shot, zero_seconds = run(ZERO_SHOT)
print(f"macro F1 {f1_score(sample.label, zero_shot, average='macro'):.3f}, {1000 * zero_seconds / len(sample):.0f} ms/text")

## 2. With a Few Examples

Add two training examples per intent to the prompt (`train.groupby("label").sample(2, random_state=0)`). Does the F1 improve? How many tokens does the prompt have now (`response.usage.prompt_tokens`), and what would it cost through an API, at $0.10 per million input tokens, for 1 million requests?

In [ ]:
# Your code here

### Solution

In [ ]:
examples = train.groupby("label").sample(2, random_state=0)
EXAMPLES = "\n".join(f'"{row.text}" -> {row.label}' for row in examples.itertuples())
FEW_SHOT = f"{ZERO_SHOT}\n\nExamples:\n{EXAMPLES}"

few_shot, few_seconds = run(FEW_SHOT)
print(f"macro F1 {f1_score(sample.label, few_shot, average='macro'):.3f}, {1000 * few_seconds / len(sample):.0f} ms/text")

usage = client.chat.completions.create(
    model=MODEL, messages=[{"role": "system", "content": FEW_SHOT}, {"role": "user", "content": sample.text[0]}], max_tokens=1
).usage
# 1 million requests of prompt_tokens tokens, at $0.10 per million tokens:
print(usage.prompt_tokens, "prompt tokens:", f"${usage.prompt_tokens * 0.10:.0f} for 1 million requests")

# A long shared prefix (instructions + examples) is exactly what the APIs' prompt caching
# bills at a discount.

## 3. Compare Every Method

Gather in one table, on the **same** 300 requests: the macro F1 of each of yesterday's methods (`sample`'s columns) and of today's two, their training time and their time per text (`timings`). Which one would you put in production for 10 million requests a month? And for a prototype due tomorrow?

In [ ]:
# Your code here

### Solution

In [ ]:
methods = [column for column in sample.columns if column not in ("id", "label", "text")]
rows = []
for method in methods:
    timing = timings.set_index("method").loc[method]
    rows.append((method, f1_score(sample.label, sample[method], average="macro"), timing.train_s, timing.ms_per_text))
rows.append(("LLM zero-shot", f1_score(sample.label, zero_shot, average="macro"), 0, 1000 * zero_seconds / len(sample)))
rows.append(("LLM few-shot", f1_score(sample.label, few_shot, average="macro"), 0, 1000 * few_seconds / len(sample)))
pd.DataFrame(rows, columns=["method", "macro F1", "training (s)", "ms/text"]).round(3)

# In production at high volume: the fine-tuned encoder (the most accurate, a few ms per text
# on CPU). For a prototype without labelled data: the LLM, right away.

## 4. Read the Errors

Show 10 errors of the best model (the text, the expected label, the prediction). Model error, or debatable label?

In [ ]:
# Your code here

### Solution

In [ ]:
best = max(methods, key=lambda m: f1_score(sample.label, sample[m], average="macro"))
errors = sample[sample[best] != sample.label]
print(best)
errors[["text", "label", best]].head(10)